# 7.0 Manipulator Dynamics

**ELEC3901 Robotic Manipulators, Lab 7** (week 7: Manipulator Dynamics)

<br>

Read before this lab: the week 7 lecture and RVC3 Ch. 9.1. Each section below follows the lecture slides listed next to it.

### Contents

| Section | Where | Time | Lecture slides |
|---|---|---|---|
| [7.1 The OMX in the toolbox](#tb) | desk | 8 min | 13, 15-17: The equations of motion; The toolbox calls; M(q) depends on the pose; The gravity load |
| [7.2 Simulation](#sim) | desk | 4 min | 21-22: Forward dynamics; Why open loop fails |
| [7.3 Gravity is linear in the link masses](#linear) | desk | 5 min | 23-24: Gravity is linear in the parameters; Least squares |
| [7.4 On the arm: holding torques](#hold) | arm | 25 min | 6, 17: What the torque does; The gravity load |
| [7.5 On the arm: your arm's model](#fit) | arm | 25 min | 23-24: Gravity is linear in the parameters; Least squares |

**How this notebook works.** The desk sections (7.1 to 7.3) repeat the lecture with the Robotics Toolbox on your arm's model:
read the worked code and run it, then complete the short *Exercises* (a few lines with `...` blanks, checked by `try_it`, which
prints `OK` or what looks wrong). The arm sections (7.4 and 7.5) are yours: you measure your arm, fit its gravity model and test
it, using the toolbox for every calculation. Nothing here asks you to write a function. Anything marked **(Optional)** is for
when the rest is done. A **Hint** opens when you click it.

*Notation.* $\mathbf{q}$ joint angles (rad), $\dot{\mathbf{q}}$ rates, $\ddot{\mathbf{q}}$ accelerations, $\boldsymbol{\tau}$ the
torque each servo **applies** (N m). Pitch is positive **nose down** as in every lab, so gravity pulls the arm in the positive
direction and the servos hold it with *negative* torque. $\mathbf{M}(\mathbf{q})$ is the inertia matrix, $\mathbf{C}\dot{\mathbf{q}}$
the Coriolis and centripetal torques, $\mathbf{g}(\mathbf{q})$ the gravity torques (lecture slide 13). Equations are numbered
(1), (2) through the notebook.

**Checkpoint for sign-off:** The printed checkpoint line: the link masses from your readings against ROBOTIS's, your servos' torque scales, the error at new poses and the friction band

This lab needs only the toolbox and the provided modules (`scripts/`); the earlier labs' functions are still imported from `omx_ref.py` if you want them.

> **Safety.** Keep the workspace clear of hands and cables before enabling torque, and set a conservative profile velocity
> (`arm.set_profile(60, 20)` or lower) before commanding motion. If anything unexpected happens, press the power isolator on the
> bench first, then think. Torque off (`arm.torque(False)`) before touching the arm.

The imports. `omx_dyn()` is the toolbox model of the OMX-F with its link masses and inertias (ROBOTIS's figures, the same as the digital twin's); `OMXSim` is the twin, used at the desk to compare with. Section 7.4 connects to the arm.

In [ ]:
import sys, os
# Repository layout: weekNN/ (notebooks), scripts/ (helper modules), data/ (your measurements, shared by every week).
for _up in ('.', '..', os.path.join('..', '..')):
    if os.path.isdir(os.path.join(_up, 'scripts')):
        _root = os.path.abspath(_up); sys.path.insert(0, os.path.join(_root, 'scripts'))
        os.makedirs(os.path.join(_root, 'data'), exist_ok=True); os.chdir(os.path.join(_root, 'data'))   # files save/load here
        break
import numpy as np, matplotlib.pyplot as plt, time, cv2
import roboticstoolbox as rtb
from spatialmath import SE3, SO3, SE2, UnitQuaternion
from spatialmath.base import trplot, tranimate
from omx_model import omx_robot, omx_ik, Q_LIM, T_EC, P_TIP_E, mount_tilt, look_down_pitch
from omx import OMX
from omx_camera import Camera, tip_in_mat, predicted_T_CM, pose_error
from omx_calib import small_T
from labcheck import check, try_it
NOM = dict(d1=0.0975, l2z=0.11315, l2x=0.0415, l3=0.162, l4=0.123)   # nominal link parameters (m)
from omx_ref import R_to_rpy, camera_ik, chain_T_BM, condition_number, dh_table, error_stats, fk_dh, inv_T, jacobian_numeric, make_T, manipulability, mount_from_views, move_metrics, omx_fk, omx_ik, planar_2r, plot_error_field, revolute_jacobian, rot2, rot_about, rotx, roty, rotz, rpy_to_R, se2, static_torque, step_log, straight_line_joints, tracking_rms, transform_point, transform_points, transl2, trot2   # reference versions of PREVIOUS labs' tasks; your own definitions below override them
np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
omx = omx_robot()              # Robotics Toolbox model (nominal)
T_EC_CAD = T_EC   # the design camera mount (ROBOTIS CAD: optical axis 38 deg below the tool). Each arm's camera sits a little differently: Lab 2
                  # measures yours, and from Lab 3 on the MOUNT cell re-measures it every session and replaces T_EC
HOVER = 0.09      # tip hover height with the tool vertical: the OMX-F reaches r ~ 0.24 m there (touching the mat: r ~ 0.28 m)
CAM_HOVER = 0.15  # tip height for camera measurements, tool pitched with look_down_pitch(T_EC) so the camera looks straight down:
                  # the camera is then ~180 mm up, enough to see the marker ring around the clear zone

In [ ]:
from omx_model import omx_dyn, HOME_Q, LINK_DYN, ARMATURE
from omx_mujoco import OMXSim
import copy, json
robot = omx_dyn()                                    # the OMX with masses: rne, inertia, coriolis, gravload and fdyn work on it
twin = OMXSim(view=False, realtime=False)            # the digital twin, for comparison at the desk (no window, simulated time)
print(robot)
print('link masses (kg):', np.round([link.m for link in robot.links[:4]], 4))

<br>

<a id='tb'></a>

### 7.1 The OMX in the toolbox (desk)
---

The lecture's equations of motion for any arm (slide 13):

$$ \mathbf{M}(\mathbf{q})\,\ddot{\mathbf{q}} + \mathbf{C}(\mathbf{q},\dot{\mathbf{q}})\,\dot{\mathbf{q}} + \mathbf{g}(\mathbf{q}) = \boldsymbol{\tau} \qquad (1) $$

The toolbox computes (1) for any arm whose links have masses: `rne(q, qd, qdd)` the whole torque, `inertia(q)`, `coriolis(q, qd)`
and `gravload(q)` the terms (slide 15). `omx_dyn()` is the D-H model of Lab 3 (`omx_dh()`) with the masses added: the same joint
angles and the same tool point as `omx_robot()`.

**Your turn: the OMX holding itself up.** Reaching out level (tool horizontal, 250 mm out, 120 mm up), how much torque does each
joint need just to hold still, and which joint works hardest? The twin computes the same thing with its own physics engine.

In [ ]:
# --- Exercise 7.1a: the holding torques reaching out level -----------------------------------------------------------
q_level = omx_ik([0.25, 0.0, 0.12], 0.0)             # tool horizontal
tau_level = ...                    # the holding torque of each joint (N m, 4 values): the toolbox's gravity term
hardest = ...                      # the joint (1 to 4) that needs the most torque
try_it('holding torques reaching out (N m)', tau_level, twin.gravity_torque(q_level), tol=1e-6)
try_it('the hardest-working joint', hardest, 1 + int(np.argmax(np.abs(twin.gravity_torque(q_level)))));

<details><summary><b>Hint</b></summary>

`robot.gravload(q)` is $\mathbf{g}(\mathbf{q})$ in (1): the torques at rest. `np.argmax(np.abs(...))` finds the largest in size;
joints are numbered from 1, Python's indices from 0.

</details>

$\mathbf{M}(\mathbf{q})$ changes with the pose (slide 16): on the Panda, joint 1's inertia $M_{11}$ grew 5.6 times as the arm reached
out. The OMX's links are light, and each joint also turns its **motor's** inertia through its gearbox (the *armature*, `ARMATURE` =
0.01 kg m² per joint in the model; week 8 explains where it comes from). It adds to the diagonal of $\mathbf{M}$ at every pose.

In [ ]:
# --- worked case: M(q) at home and reaching out, from the toolbox; the twin agrees -----------------------------------
M_home = robot.inertia(HOME_Q)
M_level = robot.inertia(q_level)
print('M at home (kg m^2):'); print(np.round(M_home, 4))
print('the twin agrees:', np.allclose(M_home, twin.inertia(HOME_Q)))
ratio = M_level[0, 0] / M_home[0, 0]
ratio_links = (M_level[0, 0] - ARMATURE) / (M_home[0, 0] - ARMATURE)
print(f'joint 1 is {ratio:.2f} times harder to swing reaching out; the links alone {ratio_links:.1f} times: on a light arm the motor dominates')

**(Optional) A payload** (slide 19) is part of the last link: add its mass to that link, and move the link's centre of mass towards it. The
toolbox's `payload(m, p)` *replaces* the last link's own mass instead (read what a function does), so do it by hand. Here 50 g held
at the tool point, which is the origin of the last link's D-H frame (its $a = l_4$ puts the frame at the tool).

In [ ]:
# --- (Optional) Exercise 7.1b: 50 g at the tool point, reaching out level --------------------------------------------
loaded = copy.deepcopy(robot)                        # a copy, so robot keeps its own masses
last = loaded.links[3]                               # joint 4's link: the wrist, gripper and camera
p_tool = np.array([0.0, 0.0, 0.0])                   # the tool point in that link's frame (m): its origin
m_pay = 0.05
r_new = ...                        # the new centre of mass: the mass-weighted average of last.r and p_tool
m_new = ...                        # the new mass
if r_new is not ...:
    last.r = r_new
    last.m = m_new
extra = ...                        # the extra holding torque reaching out (N m, 4 values): loaded minus robot
J = robot.jacob0(q_level)[:3]                        # another way: statics (week 5), tau = J^T f for the force the servos resist
try_it('extra holding torque for 50 g (N m)', extra, J.T @ np.array([0, 0, m_pay * 9.81]), tol=1e-6);

<details><summary><b>Hint</b></summary>

The centre of mass of the two together is $(m_{\mathrm{link}}\mathbf{r}_{\mathrm{link}} + m\,\mathbf{p})/(m_{\mathrm{link}} + m)$. Work
out `r_new` from the link's *old* mass `last.m`. The extra torque is the difference of the two `gravload`s.

</details>

<br>

<a id='sim'></a>

### 7.2 Simulation (desk)
---

Forward dynamics (slide 21) solves (1) for $\ddot{\mathbf{q}} = \mathbf{M}^{-1}(\boldsymbol{\tau} - \mathbf{C}\dot{\mathbf{q}} -
\mathbf{g})$ and integrates it: `fdyn`. With no motor torque, from home, the arm falls. The twin does the same with MuJoCo, and it
also models the servos' gearbox **friction** (0.1 N m per joint), which the toolbox model leaves out.

In [ ]:
# --- worked case: release the arm at home with no motor torque, 0.3 s, the toolbox against the twin ----------------------
fall = robot.fdyn(0.3, HOME_Q, Q=lambda r, t, q, qd: np.zeros(4), dt=0.005,
                  solver_args={'max_step': 0.005})                                # the toolbox: no friction, 5 ms steps
twin.set_q(HOME_Q)
twin.torque(False)
t_twin = []
q_twin = []
for k in range(60):                                  # 60 steps of 5 ms of simulated time
    twin.step(5)
    t_twin.append(0.005 * (k + 1))
    q_twin.append(twin.read_q())
q_twin = np.array(q_twin)
twin.torque(True)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for j in (1, 2, 3):
    ax[0].plot(fall.t, np.degrees(fall.q[:, j] - HOME_Q[j]), label=f'joint {j + 1}, toolbox')
    ax[1].plot(t_twin, np.degrees(q_twin[:, j] - HOME_Q[j]), label=f'joint {j + 1}, twin')
for a_ in ax:
    a_.set_xlabel('t (s)')
    a_.set_ylabel('q - q_home (deg)')
    a_.grid(alpha=0.3)
    a_.legend(fontsize=8)
ax[0].set_title('toolbox fdyn (no friction)')
ax[1].set_title('digital twin (gearbox friction 0.1 N m)')
plt.tight_layout()
plt.show()
print('joint changes after 0.3 s (deg), toolbox:', np.round(np.degrees(fall.q[-1] - HOME_Q), 1), '| twin:', np.round(np.degrees(q_twin[-1] - HOME_Q), 1))
print('gravity at home (N m):', np.round(robot.gravload(HOME_Q), 3), '- below the friction on joints 2 and 4, so in the twin they stay put')
print('the twin also stops joint 3 at its limit (96 deg) after about 0.24 s; the toolbox model has no joint limits')

<br>

<a id='linear'></a>

### 7.3 Gravity is linear in the link masses (desk)
---

Slide 23: the gravity torques are linear in the masses. Give link $k$ a mass of 1 kg (keeping its centre of mass where it is) and
every other link none: `gravload` then gives that link's **column** $\mathbf{a}_k(\mathbf{q})$, and for the real masses

$$ \mathbf{g}(\mathbf{q}) = \sum_k m_k\,\mathbf{a}_k(\mathbf{q}) = \mathbf{A}(\mathbf{q})\,\mathbf{m} \qquad (2) $$

The toolbox builds the regressor $\mathbf{A}(\mathbf{q})$ for you; nothing is derived by hand.

In [ ]:
# --- given: the columns of A(q), one link at a time, from the toolbox ----------------------------------------------------
def gravity_columns(robot, q):
    # A(q), 4 x 4: column k is gravload with link k's mass set to 1 kg and every other link's to 0 (centres of mass kept)
    unit = copy.deepcopy(robot)
    A = np.zeros((4, 4))
    for k in range(4):
        for i in range(4):
            unit.links[i].m = 1.0 if i == k else 0.0
        A[:, k] = unit.gravload(q)
    return A

A_home = gravity_columns(robot, HOME_Q)
print('A(q) at home (N m per kg):'); print(np.round(A_home, 4))
print("column 1 is all zero: link 1 turns about a vertical axis, so gravity never loads it, and no torque can reveal its mass")

In [ ]:
# --- Exercise 7.3a: the columns times the masses give gravload ---------------------------------------------------------
m_model = ...                      # the four link masses of robot (kg), as an array
g_from_columns = ...               # equation (2) at q_level: A(q_level) times the masses
try_it('A(q) m at q_level (N m)', g_from_columns, robot.gravload(q_level), tol=1e-9);

So with torques measured at many poses, stacked as in slide 24, least squares finds the masses: $\hat{\mathbf{m}} =
\mathrm{argmin}\,\lVert \mathbf{A}\mathbf{m} - \boldsymbol{\tau} \rVert$ (`np.linalg.lstsq`). Links 2 to 4 only: link 1's column is
zero. That is the rest of this lab, on your arm.

<br>

<a id='hold'></a>

### 7.4 On the arm: holding torques (arm)
---

From here on you need the arm, or the digital twin (`SIMULATE = True` in the next cell). The camera is not used.

**What the servos tell you.** `arm.read_torque()` returns each joint's torque estimate (N m), and it is only an estimate. Joint 4 (an
XL330) measures its current; joints 1 to 3 (XL430s) have no current sensor and report a *load*, inferred from the drive they apply.
Both are scaled by the servo's stall figures (1.4 N m and 0.52 N m). The gearbox's friction adds a band: a held joint reads a little
differently depending on which way it last moved, because friction helps the servo hold it one way and fights it the other. So
you reach every pose from both sides and average: the friction cancels, and the difference measures the band. How close the
estimate is to the true torque, you find out in Section 7.5.

In [ ]:
# --- connect to the arm --------------------------------------------------------------------------------------------
# Safe to re-run: an existing connection is closed first, so the COM port is released before it is opened again.
# SIMULATE = True runs the MuJoCo digital twin (omx_mujoco.py) instead of the real arm: same API, a rendered
# wrist camera, and optional hidden model errors so you can test calibration against ground truth.
SIMULATE = False
PORT = 'auto'                  # finds the OpenRB-150 by its USB id; or set it, e.g. 'COM4' (Device Manager) / '/dev/ttyACM0'
# --- which arm are you on today? Read the asset label on the base. Everything you measure is tagged with it,
#     because zero offsets, camera intrinsics and the camera mount are all properties of THIS arm.
ARM_ID = 'OMX-00'            # EDIT every session
if 'arm' in globals():
    try:
        arm.close()
    except Exception as e:
        print('closing the previous connection:', e)
    del arm
if SIMULATE:
    from omx_mujoco import OMXSim, SimCamera
    arm = OMXSim(view=True)    # view=False if no display; set kin_error=..., joint_offsets=... to hide errors
    # Video: arm.record('run.mp4') writes the wrist camera and a scene camera side by side (same size) while the arm
    # moves; arm.stop_recording() when done.
else:
    arm = OMX(PORT)
print('connected:', 'digital twin' if SIMULATE else arm.port_name, '| arm', ARM_ID)

In [ ]:
# --- given: 12 poses to fit and 3 to test, tool clear of the mat (checked with the toolbox before the arm goes anywhere) ---
rng = np.random.default_rng(7)
candidates = np.radians(rng.uniform([0, -50, 10, -40], [0, 10, 80, 60], (300, 4)))     # joint 1 fixed at 0
safe = []
for q in candidates:
    p = robot.fkine(q).t                                                                  # the tool point (m)
    if p[2] >= 0.06 and np.hypot(p[0], p[1]) >= 0.12:                                    # 60 mm up, 120 mm out from the base axis
        safe.append(q)
Q_fit = np.array(safe[:12])
Q_test = np.array(safe[12:15])
print(len(Q_fit), 'poses to fit,', len(Q_test), 'to test; lowest tool point', round(1e3 * min(robot.fkine(q).t[2] for q in Q_fit)), 'mm')

**Your turn: measure.** For each pose: come to it from 10° short on each pitch joint, wait a second for it to settle, and average 20
readings of `arm.read_torque()`; then the same from 10° beyond. About a minute and a half for the 15 poses; keep the workspace
clear.

In [ ]:
# --- Exercise 7.4a: the holding torques at Q_fit and Q_test, from both sides ---------------------------------------------
SIDE = np.radians([0, 10, 10, 10])                   # come to each pose from q - SIDE, then from q + SIDE

def hold(arm, Q):
    # given the poses Q (N x 4), returns the holding torques reached from one side and from the other (each N x 4, N m)
    one_side = []
    other_side = []
    for q in Q:
        for start, store in ((q - SIDE, one_side), (q + SIDE, other_side)):
            arm.move_q(start, wait=True)
            ...                    # move to q and wait until it is there
            time.sleep(1.0)
            readings = []
            for k in range(20):
                ...                # append one arm.read_torque() reading
            store.append(...)      # the average of the 20 readings
    return np.array(one_side), np.array(other_side)

arm.torque(True)
arm.set_profile(40, 10)
arm.home(wait=True)
TAU_a, TAU_b = hold(arm, Q_fit)
TEST_a, TEST_b = hold(arm, Q_test)
arm.home(wait=True)
if TAU_a.dtype == object:                            # hold() not finished yet
    TAU_fit = TAU_a
    TAU_test = TEST_a
else:
    TAU_fit = (TAU_a + TAU_b) / 2                    # the friction cancels in the average
    TAU_test = (TEST_a + TEST_b) / 2
np.savez('lab7_holding.npz', arm_id=ARM_ID, Q_fit=Q_fit, TAU_a=TAU_a, TAU_b=TAU_b, Q_test=Q_test, TEST_a=TEST_a, TEST_b=TEST_b)
try_it('measured torques (12 x 4)', TAU_fit.shape if TAU_fit.dtype != object and TAU_fit.size > 0 else ..., (12, 4))
print('saved lab7_holding.npz')

In [ ]:
# the measured holding torques against the toolbox model's, joint by joint: each side, and the average (after 7.4a)
G_model = np.array([robot.gravload(q) for q in Q_fit])
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for j in (1, 2, 3):
    if TAU_fit.ndim < 2:
        print('measure first: Exercise 7.4a')
        break
    ax[j - 1].plot(G_model[:, j], TAU_a[:, j], 'v', ms=4, alpha=0.6, label='one side')
    ax[j - 1].plot(G_model[:, j], TAU_b[:, j], '^', ms=4, alpha=0.6, label='other side')
    ax[j - 1].plot(G_model[:, j], TAU_fit[:, j], 'o', label='average')
    lim = [min(G_model[:, j].min(), TAU_a[:, j].min(), TAU_b[:, j].min()) - 0.02, max(G_model[:, j].max(), TAU_a[:, j].max(), TAU_b[:, j].max()) + 0.02]
    ax[j - 1].plot(lim, lim, 'k:', lw=1)
    ax[j - 1].set_xlabel('model gravload (N m)')
    ax[j - 1].set_ylabel('measured (N m)')
    ax[j - 1].set_title(f'joint {j + 1}')
    ax[j - 1].grid(alpha=0.3)
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

<br>

<a id='fit'></a>

### 7.5 On the arm: your arm's model (arm)
---

Stack equation (2) for the 12 poses, joints 2 to 4 and links 2 to 4 only (slide 24): a $36 \times 3$ matrix of columns from the
toolbox, and 36 measured torques (the averages). The least-squares masses make the model's holding torques as close to your readings
as they can be; what is left over, the **residual**, is everything the model does not have.

In [ ]:
# --- Exercise 7.5a: least squares for the masses of links 2 to 4 ----------------------------------------------------------
rows = []
rhs = []
for q, tau in zip(Q_fit, TAU_fit):
    A = gravity_columns(robot, q)
    rows.append(...)               # joints 2-4, links 2-4: the bottom-right 3 x 3 block of A
    rhs.append(...)                # the measured torques of joints 2-4
A_all = np.vstack(rows)            # 36 x 3
b_all = np.concatenate(rhs)        # 36
m_fit = ...                        # the least-squares masses (kg, 3 values): np.linalg.lstsq(A_all, b_all, rcond=None)[0]
residual_rms = ...                 # the RMS of A_all m_fit - b_all (N m)
try_it('masses of links 2-4 (kg)', m_fit, np.linalg.pinv(A_all) @ b_all, tol=1e-9)
try_it('RMS residual (N m)', residual_rms, np.sqrt(np.mean((A_all @ (np.linalg.pinv(A_all) @ b_all) - b_all) ** 2)), tol=1e-9);

<details><summary><b>Hint</b></summary>

`A[1:4, 1:4]` takes rows 2 to 4 and columns 2 to 4 (Python counts from 0). `np.linalg.lstsq(...)` returns several things; the first,
`[0]`, is the solution.

</details>

In [ ]:
# the masses straight from the servos' readings, against ROBOTIS's (weighed parts and CAD)
m_robotis = np.array([link.m for link in robot.links[1:4]])
print('links 2-4 (g): ROBOTIS', np.round(1e3 * m_robotis), '| from your readings', np.round(1e3 * m_fit),
      f'| RMS residual {residual_rms:.3f} N m')

**Is it the arm or the sensor?** On a real OMX the masses from the readings come out far from ROBOTIS's, some too light and some too
heavy, yet the residual is small. The links have not changed weight: the readings are not torques. Section 7.4 said so: an XL430
reports a *load* inferred from its drive, and the XL330's gearbox (288:1) holds much of the load itself, so its motor current is
small. If each joint's reading is a fixed fraction $k_j$ of the true torque, the masses absorb a different fraction for every joint
and come out wrong, while the fit still looks good.

So turn the question round. Take the masses as known (ROBOTIS's, in `robot`) and fit, for each pitch joint, the one number $k_j$ in
$$\tau_{\text{read}, j} = k_j \, g_j(\mathbf{q}) \tag{3}$$
over the 12 poses: least squares again, with a single column, the model's holding torques $g_j$ from `gravload`.

In [ ]:
# --- Exercise 7.5b: each servo's torque scale ---------------------------------------------------------------------------
G_fit = np.array([robot.gravload(q) for q in Q_fit])     # the model's holding torques at the 12 poses (12 x 4)
scale = [1.0]                      # joint 1 carries no gravity: nothing to measure
for j in (1, 2, 3):
    g_j = ...                      # column j of G_fit
    r_j = ...                      # column j of TAU_fit (your averaged readings)
    k_j = ...                      # least squares with one column: np.linalg.lstsq(g_j.reshape(-1, 1), r_j, rcond=None)[0][0]
    scale.append(k_j)
_k = lambda: [1.0] + [float(G_fit[:, j] @ TAU_fit[:, j] / (G_fit[:, j] @ G_fit[:, j])) for j in (1, 2, 3)]
try_it('torque scale of joints 1-4', scale, _k(), tol=1e-9);

<details><summary><b>Hint</b></summary>

With one column the least-squares answer is $k_j = \sum g_j \tau_j / \sum g_j^2$; `lstsq` gives the same. A scale of 1 would mean the
servo reads the true torque; in the digital twin it does.

</details>

**Checking it.** A fit always looks good on its own data. The test: the 3 poses of `Q_test`, which the fit never saw. Predict what the
servos will read there, $k_j g_j(\mathbf{q})$, and compare with taking the readings as torques ($k_j = 1$). Then the friction band: the
difference between the two sides, divided by the scale to put it in true N m.

In [ ]:
# --- Exercise 7.5c: the scales at new poses -----------------------------------------------------------------------------
G_test = np.array([robot.gravload(q) for q in Q_test])   # the model's holding torques at the 3 new poses (3 x 4)
pred_scaled = ...                  # what the servos should read there: G_test times scale (joint by joint)
err_scaled = ...                   # the RMS of pred_scaled - TAU_test over joints 2-4 (columns 1 to 3) (N m)
err_raw = ...                      # the same with G_test itself: the readings taken as torques
_rms = lambda P: np.sqrt(np.mean((P[:, 1:] - TAU_test[:, 1:]) ** 2))
try_it('scaled model at new poses: RMS error (N m)', err_scaled, _rms(G_test * np.array(_k())), tol=1e-9)
try_it('readings taken as torques: RMS error (N m)', err_raw, _rms(G_test), tol=1e-9);

In [ ]:
# --- Exercise 7.5d: the friction band, in true N m ------------------------------------------------------------------------
band_read = ...                    # the mean over Q_fit of TAU_a minus TAU_b, per joint (4 values, in reading units)
band = ...                         # band_read divided by scale: the friction the gearbox holds, in N m
try_it('friction band (N m)', band, np.mean(TAU_a - TAU_b, axis=0) / np.array(_k()), tol=1e-9);

Where the band is as large as the holding torque, the gearbox holds the joint by itself: switch the torque off there and it stays
put, as joints 2 and 4 did in the twin in Section 7.2. Lab 9 uses your scales to turn the servos' readings into torques.

In [ ]:
# save your arm's gravity model (Lab 9 uses it), then the checkpoint
json.dump({'arm_id': ARM_ID, 'link_masses_kg': [float(link.m) for link in robot.links], 'torque_scale': [float(k_) for k_ in scale],
           'friction_band_Nm': [float(b_) for b_ in band], 'rms_test_Nm': float(err_scaled)}, open('lab7_gravity_model.json', 'w'), indent=1)
# --- checkpoint: show this line to a demonstrator ------------------------------------------------------------------
print(f'ARM {ARM_ID} | masses from readings {np.round(1e3 * m_fit).astype(int).tolist()} g (ROBOTIS {np.round(1e3 * m_robotis).astype(int).tolist()}) '
      f'| torque scales {np.round(scale[1:], 2).tolist()} | new poses {err_scaled:.3f} N m (as torques {err_raw:.3f}) '
      f'| friction band {np.round(band[1:], 3).tolist()} N m | saved lab7_gravity_model.json')

**The toolbox calls in this lab:** `rne(q, qd, qdd)` (the torque for a motion), `gravload(q)`, `inertia(q)`, `coriolis(q, qd)` (the
terms), `fdyn(T, q0, Q=...)` (simulation), `fkine(q)` (the pose), and the regressor's columns from `gravload` with one link's mass
at a time; then `np.linalg.lstsq` for the fits.

## Before you leave
1. Park the arm (`arm.rest()`: the folded rest pose, where it stays put with torque off), then `arm.torque(False)` and `arm.close()`.
   Never switch torque off at home or mid-task: the arm drops under gravity.
2. Switch off the bench power supply and tidy cables.
3. Show your checkpoint output to a demonstrator for sign-off.
4. Commit this notebook and any data files to your own LABS repository and **push** (Sync Changes): later labs read them. The assessment repository (issued in week 6) starts fresh; copy across any code you want to reuse.

In [ ]:
arm.set_profile(40, 10); arm.rest(wait=True)     # folded rest pose: stable with torque off
arm.torque(False)
arm.close()
if globals().get('cam') is not None: cam.close()